# Lecture 6 | EDA & Data Quality: Unusual Values and Missing Data
**Course:** Data Analytics for Managers Using Python  |  **Module 2** (LO2: Clean, summarize, visualize and interpret business datasets)

**Case:** SkyBridge Airways - Can we trust our own numbers?

**Key question:** *How should managers treat unusual values in business datasets?*

**Today's steps**
1. Load Clean and Unclean data
2. Find blank cells (Before heatmap)
3. Compare descriptive statistics
4. See the data over time
5. Boxplots for every variable
6. Find outliers with the IQR rule
7. Decide: Retain, Remove, Cap or Investigate
8. Fix blanks 4 ways: `dropna()`, mean, median, mode
9. After heatmap and comparison
10. AI support for explaining decisions

## Step 0: Import the tools

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

sns.set_style("whitegrid")

## Step 1: Load the data
Run the cell, click **Choose files**, and select `SkyBridge_Airline_Data.xlsx`.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
clean   = pd.read_excel("SkyBridge_Airline_Data.xlsx", sheet_name="Clean_Data")
unclean = pd.read_excel("SkyBridge_Airline_Data.xlsx", sheet_name="Unclean_Data")

# These are the 9 numeric columns we will study
num_cols = ["Passengers", "Ticket_Price", "Delay_Minutes", "Flight_Duration_Min", "Fuel_Cost",
            "Ancillary_Revenue", "Baggage_Complaints", "Customer_Rating", "Crew_Overtime_Hours"]

print("Clean data  :", clean.shape)
print("Unclean data:", unclean.shape)
unclean.head()

## Step 2: Count the blank (missing) cells

In [ ]:
unclean.isnull().sum()

In [ ]:
# What % of cells are blank? And how many rows have at least one blank?
total_blank = unclean.isnull().sum().sum()
print("Blank cells :", total_blank)
print("Blank %     :", round(total_blank / unclean.size * 100, 1), "%")
print("Rows with at least one blank:", unclean.isnull().any(axis=1).sum(), "out of", len(unclean))

### BEFORE heatmap: where are the blanks?
Red = blank cell, grey = filled cell. Look for patterns (e.g. many blanks in a row).

In [ ]:
colors = ListedColormap(["#d9d9d9", "#d62728"])

plt.figure(figsize=(10, 6))
sns.heatmap(unclean.isnull(), cmap=colors, cbar=False, yticklabels=False)
plt.title("BEFORE: Blank cells in Unclean data (red = blank)")
plt.show()

## Step 3: Compare descriptive statistics (Clean vs Unclean)

In [ ]:
compare = pd.DataFrame({
    "Clean mean":     clean[num_cols].mean(),
    "Unclean mean":   unclean[num_cols].mean(),
    "Clean median":   clean[num_cols].median(),
    "Unclean median": unclean[num_cols].median(),
    "Clean std":      clean[num_cols].std(),
    "Unclean std":    unclean[num_cols].std(),
})
compare.round(1)

In [ ]:
# Full summary of the unclean data: look at min and max carefully
unclean[num_cols].describe().round(1)

**Think:** Which variables have a `max` or `min` that looks impossible for an airline?
Which has a **mean** far from its **median**? (That is a warning sign.)

## Step 4: Time series view
One flight per day, so we can plot each variable over time.

In [ ]:
ts_clean   = clean.set_index("Date")
ts_unclean = unclean.set_index("Date")

fig, axes = plt.subplots(2, 1, figsize=(11, 7))

ts_clean["Delay_Minutes"].plot(ax=axes[0], color="green", title="CLEAN: Delay (minutes) over time")
ts_unclean["Delay_Minutes"].plot(ax=axes[1], color="red",   title="UNCLEAN: Delay (minutes) over time")

plt.tight_layout()
plt.show()

In [ ]:
# Another variable: Ticket price over time
fig, axes = plt.subplots(2, 1, figsize=(11, 7))

ts_clean["Ticket_Price"].plot(ax=axes[0], color="green", title="CLEAN: Ticket price over time")
ts_unclean["Ticket_Price"].plot(ax=axes[1], color="red",   title="UNCLEAN: Ticket price over time")

plt.tight_layout()
plt.show()

**Think:** Gaps in the red line are blank cells. Spikes and dips are unusual values. Which spikes look like typing errors and which look like real events?

## Step 5: Boxplots for every variable
A boxplot shows the median (line), the middle 50% of data (box) and unusual values (dots beyond the whiskers).

In [ ]:
def show_boxplots(data, title):
    fig, axes = plt.subplots(3, 3, figsize=(13, 10))
    axes = axes.flatten()
    for i, col in enumerate(num_cols):
        sns.boxplot(y=data[col], ax=axes[i], color="skyblue")
        axes[i].set_title(col)
        axes[i].set_ylabel("")
    fig.suptitle(title, fontsize=15)
    plt.tight_layout()
    plt.show()

show_boxplots(clean, "CLEAN data: boxplots")

In [ ]:
show_boxplots(unclean, "UNCLEAN data: boxplots")

**Think:** In which variables did the boxplot get squashed because of one extreme dot? Compare each panel with the clean version.

## Step 6: Find outliers with the IQR rule
**IQR rule:**  
- Q1 = 25th percentile, Q3 = 75th percentile, IQR = Q3 - Q1  
- Lower fence = Q1 - 1.5 x IQR  
- Upper fence = Q3 + 1.5 x IQR  
- Any value outside the fences is flagged as a possible outlier.

First, one variable step by step: **Ticket_Price**

In [ ]:
col = "Ticket_Price"

Q1  = unclean[col].quantile(0.25)
Q3  = unclean[col].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("Q1 =", Q1, "| Q3 =", Q3, "| IQR =", IQR)
print("Lower fence =", lower, "| Upper fence =", upper)

outliers = unclean[(unclean[col] < lower) | (unclean[col] > upper)]
outliers[["Date", "Flight_ID", "Route", col]]

Now the same rule for **all** variables, in one table:

In [ ]:
rows = []
for col in num_cols:
    Q1  = unclean[col].quantile(0.25)
    Q3  = unclean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    count = ((unclean[col] < lower) | (unclean[col] > upper)).sum()
    rows.append([col, round(lower, 1), round(upper, 1), count])

outlier_table = pd.DataFrame(rows, columns=["Variable", "Lower fence", "Upper fence", "Outliers found"])
outlier_table

Which exact rows and values were flagged? (This list is what a manager must investigate.)

In [ ]:
flagged = []
for col in num_cols:
    Q1  = unclean[col].quantile(0.25)
    Q3  = unclean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    bad = unclean[(unclean[col] < lower) | (unclean[col] > upper)]
    for idx, row in bad.iterrows():
        flagged.append([row["Date"].date(), row["Flight_ID"], col, row[col]])

flagged_df = pd.DataFrame(flagged, columns=["Date", "Flight_ID", "Variable", "Value"])
flagged_df

## Step 7: Business interpretation - what do we DO with an unusual value?

| Decision | When to use | Airline example |
|---|---|---|
| **Retain** | Value is real and part of normal business | A busy Friday with 180 passengers |
| **Remove** | Value is impossible or wrong and cannot be fixed | 215 passengers on a 180-seat aircraft |
| **Cap** | Value is real but so extreme that it distorts the average | A very high ancillary revenue day |
| **Investigate** | Not sure if error or real event; check with the business | A 380-minute delay (fog or typing mistake?) |

**Golden rule:** *An outlier is a question, not an answer.* Ask "Is it possible? Is it explainable?" before touching it.

**Check the evidence:** a 380-minute delay. Do other variables on that same day also look unusual? If yes, it is probably a real event.

In [ ]:
unclean[unclean["Delay_Minutes"] > 100]

**Capping** (also called winsorizing): replace values beyond the fences with the fence value.

In [ ]:
col = "Ancillary_Revenue"

Q1  = unclean[col].quantile(0.25)
Q3  = unclean[col].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

capped = unclean[col].clip(lower, upper)

print("Mean before capping:", round(unclean[col].mean()))
print("Mean after capping :", round(capped.mean()))
print("Max before:", unclean[col].max(), "| Max after:", round(capped.max()))

**Removing impossible values:** turn them into blanks, so they can be treated like other missing values.

In [ ]:
unclean2 = unclean.copy()

unclean2.loc[unclean2["Passengers"] > 180, "Passengers"] = None            # more than seats
unclean2.loc[unclean2["Delay_Minutes"] < 0, "Delay_Minutes"] = None         # negative delay
unclean2.loc[unclean2["Customer_Rating"] > 5, "Customer_Rating"] = None     # rating above 5

print("Blank cells before:", unclean.isnull().sum().sum())
print("Blank cells after :", unclean2.isnull().sum().sum())

## Step 8: Fix the blanks - four ways
Fix the blank cells in the **unclean** data and compare.

| Method | Command | Idea |
|---|---|---|
| 1. Drop | `dropna()` | Delete every row that has a blank |
| 2. Mean | `fillna(mean)` | Fill with the average |
| 3. Median | `fillna(median)` | Fill with the middle value |
| 4. Mode | `fillna(mode)` | Fill with the most common value |

### Method 1: `dropna()`

In [ ]:
df_drop = unclean.dropna()

print("Rows before:", len(unclean))
print("Rows after :", len(df_drop))
print("Data lost  :", round((1 - len(df_drop) / len(unclean)) * 100, 1), "%")

### Method 2: Fill with the **mean**

In [ ]:
df_mean = unclean.copy()
df_mean[num_cols] = df_mean[num_cols].fillna(df_mean[num_cols].mean())

# Route is text, so a mean is not possible. Use the most common route.
df_mean["Route"] = df_mean["Route"].fillna(df_mean["Route"].mode()[0])

print("Blanks left:", df_mean.isnull().sum().sum())

### Method 3: Fill with the **median**

In [ ]:
df_median = unclean.copy()
df_median[num_cols] = df_median[num_cols].fillna(df_median[num_cols].median())

# Route is text, so use the most common route
df_median["Route"] = df_median["Route"].fillna(df_median["Route"].mode()[0])

print("Blanks left:", df_median.isnull().sum().sum())

### Method 4: Fill with the **mode** (most common value)

In [ ]:
df_mode = unclean.copy()

for col in df_mode.columns:
    df_mode[col] = df_mode[col].fillna(df_mode[col].mode()[0])

print("Blanks left:", df_mode.isnull().sum().sum())

### AFTER heatmaps: are the blanks gone?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 6))

sns.heatmap(df_drop.isnull(),   cmap=colors, cbar=False, yticklabels=False, ax=axes[0])
axes[0].set_title("After dropna()")

sns.heatmap(df_mean.isnull(),   cmap=colors, cbar=False, yticklabels=False, ax=axes[1])
axes[1].set_title("After Mean fill")

sns.heatmap(df_median.isnull(), cmap=colors, cbar=False, yticklabels=False, ax=axes[2])
axes[2].set_title("After Median fill")

sns.heatmap(df_mode.isnull(),   cmap=colors, cbar=False, yticklabels=False, ax=axes[3])
axes[3].set_title("After Mode fill")

plt.suptitle("AFTER: No red cells left (grey = filled)", fontsize=14)
plt.tight_layout()
plt.show()

**Notice:** `dropna()` removes red cells by removing **rows** (the heatmap gets shorter). The other three methods keep all rows.

## Step 9: Which method gives numbers closest to the truth?
The **Clean** data is our benchmark. Which method is closest to it?

In [ ]:
mean_table = pd.DataFrame({
    "Clean (truth)": clean[num_cols].mean(),
    "Drop":          df_drop[num_cols].mean(),
    "Mean fill":     df_mean[num_cols].mean(),
    "Median fill":   df_median[num_cols].mean(),
    "Mode fill":     df_mode[num_cols].mean(),
})
mean_table.round(1)

In [ ]:
median_table = pd.DataFrame({
    "Clean (truth)": clean[num_cols].median(),
    "Drop":          df_drop[num_cols].median(),
    "Mean fill":     df_mean[num_cols].median(),
    "Median fill":   df_median[num_cols].median(),
    "Mode fill":     df_mode[num_cols].median(),
})
median_table.round(1)

In [ ]:
std_table = pd.DataFrame({
    "Clean (truth)": clean[num_cols].std(),
    "Drop":          df_drop[num_cols].std(),
    "Mean fill":     df_mean[num_cols].std(),
    "Median fill":   df_median[num_cols].std(),
    "Mode fill":     df_mode[num_cols].std(),
})
std_table.round(1)

In [ ]:
# Boxplot of Delay_Minutes across all versions of the data
delay_compare = pd.DataFrame({
    "Clean":       clean["Delay_Minutes"],
    "Drop":        df_drop["Delay_Minutes"],
    "Mean fill":   df_mean["Delay_Minutes"],
    "Median fill": df_median["Delay_Minutes"],
    "Mode fill":   df_mode["Delay_Minutes"],
})

plt.figure(figsize=(10, 5))
sns.boxplot(data=delay_compare, color="skyblue")
plt.title("Delay_Minutes: Clean vs four fixing methods")
plt.show()

In [ ]:
# Same for Ticket_Price
price_compare = pd.DataFrame({
    "Clean":       clean["Ticket_Price"],
    "Drop":        df_drop["Ticket_Price"],
    "Mean fill":   df_mean["Ticket_Price"],
    "Median fill": df_median["Ticket_Price"],
    "Mode fill":   df_mode["Ticket_Price"],
})

plt.figure(figsize=(10, 5))
sns.boxplot(data=price_compare, color="lightgreen")
plt.title("Ticket_Price: Clean vs four fixing methods")
plt.show()

### What did we learn?
- **dropna()** is simple but can throw away a large share of your data.
- **Mean** is pulled by outliers (see Ticket_Price and Delay_Minutes). Use it when data is roughly symmetric and has no extreme values.
- **Median** is not pulled by outliers, so it is the safer default for skewed business data (prices, delays, revenue).
- **Mode** is best for categories (Route) and for count-type variables.
- **Order matters:** notice that Ticket_Price is still too high in *every* method, because the typing error (an extra zero) is still in the data. Deal with impossible values first (Step 7), then fill blanks (Step 8).
- *Bonus for time series:* `df.ffill()` fills a blank with the previous day's value. Try it on Fuel_Cost, which had three blank days in a row.

## Step 10: AI support for explaining outlier decisions
AI can **explain** and **draft** a justification. It cannot know your airline's facts. You, the manager, must verify.

The code below builds a ready-to-paste prompt from the outliers we found. Copy the printed text into an AI assistant (Claude, ChatGPT, Gemini).

In [ ]:
prompt = '''You are a business data analyst advising an airline operations head.
Dataset: daily flight log of SkyBridge Airways (Delhi hub), 120 days. Aircraft has 180 seats. Customer rating scale is 1 to 5.

These values were flagged as outliers by the IQR rule:
''' + flagged_df.to_string(index=False) + '''

For each flagged value:
1. Say whether it looks like a data-entry error, a genuine extreme event, or normal variation in a skewed variable.
2. Recommend Retain, Remove, Cap or Investigate, with a one-line business reason.
3. Tell me what extra evidence I should check before deciding.
Keep the answer in a table.'''

print(prompt)

**Verify before you trust AI** (write your answers in class):
1. Did the AI use the facts we gave it (180 seats, rating 1 to 5)?
2. Did it assume anything about the airline that we never told it?
3. Do you agree with every Retain / Remove / Cap / Investigate decision? Which one would you change, and why?

## Your turn (class exercise)
1. Which **two** variables are most damaged by the unusual values? Show using the mean vs median table.
2. For `Ticket_Price`, which fixing method (drop, mean, median, mode) is closest to the clean mean? Why?
3. Rerun Step 8 using `unclean2` (impossible values already removed) instead of `unclean`. What changes in the mean table?
4. Write a 5-line recommendation to the Operations Head: what to keep, what to remove, how to fill the blanks.